## Imports

In [6]:
import sys
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv("../.env")

from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from datasets import Dataset
import pandas as pd

from app.core.retrieval import retrieve
from app.core.generation import generate, create_chat_history
from app.config import settings

C:\Users\saika\AppData\Local\Temp\ipykernel_38816\3761250105.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\saika\AppData\Local\Temp\ipykernel_38816\3761250105.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\saika\AppData\Local\Temp\ipykernel_38816\3761250105.py:7: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from

## Define test questions and references

In [7]:
testset_df = pd.DataFrame([
    {"user_input": "What is the full year FY23 revenue guidance?",
     "reference": "$31.7 - $31.8 Billion"},
    {"user_input": "What is the Q2 FY23 revenue guidance?",
     "reference": "$7.69 - $7.70 Billion"},
    {"user_input": "What is the FX impact on full year revenue?",
     "reference": "~($600M) year over year FX impact"},
    {"user_input": "What is the non-GAAP operating margin guidance?",
     "reference": "~20.4%"},
    {"user_input": "What is the GAAP operating margin guidance?",
     "reference": "~3.8%"},
    {"user_input": "What is the operating cash flow growth guidance?",
     "reference": "~21% - 22%"},
    {"user_input": "What is the current remaining performance obligation?",
     "reference": "Approximately $21.5 billion, an increase of 21% year-over-year"},
    {"user_input": "What assumptions does the guidance make about the investment portfolio?",
     "reference": "Guidance assumes no change to the value of the strategic investment portfolio"},
    {"user_input": "When is the earnings call scheduled?",
     "reference": "May 31, 2022 at 2:00 PM Pacific Time"},
    {"user_input": "What is the GAAP earnings per share guidance for Q2?",
     "reference": "($0.03) - ($0.02)"},
])

print(testset_df)

                                          user_input  \
0       What is the full year FY23 revenue guidance?   
1              What is the Q2 FY23 revenue guidance?   
2        What is the FX impact on full year revenue?   
3    What is the non-GAAP operating margin guidance?   
4        What is the GAAP operating margin guidance?   
5   What is the operating cash flow growth guidance?   
6  What is the current remaining performance obli...   
7  What assumptions does the guidance make about ...   
8               When is the earnings call scheduled?   
9  What is the GAAP earnings per share guidance f...   

                                           reference  
0                              $31.7 - $31.8 Billion  
1                              $7.69 - $7.70 Billion  
2                  ~($600M) year over year FX impact  
3                                             ~20.4%  
4                                              ~3.8%  
5                                         ~21% - 22% 

## RAG pipeline runner

In [8]:
def run_pipeline(questions, references, namespace, rerank=True):
    results = []
    chat_history = create_chat_history()

    for question, reference in zip(questions, references):
        chunks = retrieve(question, namespace=namespace, rerank=rerank)
        answer = generate(question, chunks, chat_history)
        results.append({
            "user_input": question,
            "response": answer,
            "retrieved_contexts": [c["text"] for c in chunks],
            "reference": reference
        })
    return results

## Run eval — dense only vs dense + rerank

In [9]:
namespace = "salesforcefinancial.pdf"
questions = testset_df["user_input"].tolist()
references = testset_df["reference"].tolist()

print("Running dense only...")
dense_results = run_pipeline(questions, references, namespace, rerank=False)

print("Running dense + rerank...")
rerank_results = run_pipeline(questions, references, namespace, rerank=True)

print("Done — running RAGAS eval...")

eval_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", api_key=settings.openai_api_key)
)
eval_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model=settings.embedding_model, api_key=settings.openai_api_key)
)

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

dense_dataset = Dataset.from_list(dense_results)
rerank_dataset = Dataset.from_list(rerank_results)

dense_scores = evaluate(dense_dataset, metrics=metrics, llm=eval_llm, embeddings=eval_embeddings)
rerank_scores = evaluate(rerank_dataset, metrics=metrics, llm=eval_llm, embeddings=eval_embeddings)

print("Dense only:", dense_scores)
print("Dense + Rerank:", rerank_scores)

Running dense only...
Running dense + rerank...
Done — running RAGAS eval...


C:\Users\saika\AppData\Local\Temp\ipykernel_38816\3752239430.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(
C:\Users\saika\AppData\Local\Temp\ipykernel_38816\3752239430.py:16: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(
Evaluating:   2%|▎         | 1/40 [00:06<04:09,  6.40s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generatio

Dense only: {'faithfulness': 0.8000, 'answer_relevancy': 0.8211, 'context_precision': 0.9389, 'context_recall': 1.0000}
Dense + Rerank: {'faithfulness': 0.9000, 'answer_relevancy': 0.8256, 'context_precision': 0.9667, 'context_recall': 1.0000}


## Results table

In [10]:
results_df = pd.DataFrame({
    "Configuration": ["Dense only", "Dense + Cohere rerank"],
    "Faithfulness": [
        dense_scores["faithfulness"],
        rerank_scores["faithfulness"]
    ],
    "Context Precision": [
        dense_scores["context_precision"],
        rerank_scores["context_precision"]
    ],
    "Context Recall": [
        dense_scores["context_recall"],
        rerank_scores["context_recall"]
    ],
    "Answer Relevancy": [
        dense_scores["answer_relevancy"],
        rerank_scores["answer_relevancy"]
    ]
})

print(results_df.to_string(index=False))
results_df.to_csv("eval_results.csv", index=False)

        Configuration                                       Faithfulness                                                                                                                                                    Context Precision                                     Context Recall                                                                                                                                                           Answer Relevancy
           Dense only [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0] [0.99999999995, 0.8055555555287036, 0.99999999995, 0.9999999999, 0.99999999995, 0.9999999999666667, 0.8333333332916666, 0.9999999999, 0.9999999999, 0.7499999999625] [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]   [0.9336060134463929, 0.9120727837236146, 0.9800077441949258, 0.8669650409179824, 0.8709598979018107, 0.9383552930698505, 0.76619436339598, 0.0, 1.0, 0.9432396842364442]
Dense + Cohere rerank [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0] [0.9999